## Login-Code

In [2]:
import requests
import time
from selenium import webdriver

LOGIN_URL = "https://ilias.uni-konstanz.de/login.php?client_id=ILIASKONSTANZ&cmd=force_login&lang=de"

print("Starting browser...")
driver = webdriver.Chrome()
driver.get(LOGIN_URL)

print("Browser is open! Please type your credentials into the Chrome window.")
print("Waiting for you to log in... (I will automatically detect when you are done)")

# --- THE NEW FIX: Watch the URL instead of asking for input ---
# Give you 2 minutes (120 seconds) to type your password
timeout = 120 
elapsed = 0

# Check the URL every 2 seconds. 
while "login.php" in driver.current_url and elapsed < timeout:
    time.sleep(2)
    elapsed += 2

# Once the loop finishes, let's see what happened
if "login.php" in driver.current_url:
    print("❌ Timeout! You didn't log in within 2 minutes.")
    driver.quit()
else:
    print("✅ Login detected! The URL changed. Securing cookies...")
    time.sleep(3) # Give the dashboard 3 seconds to fully load
    
    user_agent = driver.execute_script("return navigator.userAgent;")
    selenium_cookies = driver.get_cookies()

    session = requests.Session()
    session.headers.update({'User-Agent': user_agent})

    for cookie in selenium_cookies:
        session.cookies.set(cookie['name'], cookie['value'])

    print("Cookies and User-Agent successfully transferred! Closing Chrome...")
    driver.quit()

    # Test the session
    test_response = session.get("https://ilias.uni-konstanz.de/")
    if test_response.status_code == 200:
        print("🎉 Successfully logged in and ready for scraping!")
    else:
        print(f"❌ Something went wrong testing the session. Status Code: {test_response.status_code}")

Starting browser...
Browser is open! Please type your credentials into the Chrome window.
Waiting for you to log in... (I will automatically detect when you are done)
✅ Login detected! The URL changed. Securing cookies...
Cookies and User-Agent successfully transferred! Closing Chrome...
🎉 Successfully logged in and ready for scraping!


## Fetching the Course Page HTML

In [3]:
from bs4 import BeautifulSoup

# The URL of the specific course you want to scrape
COURSE_URL = "https://ilias.uni-konstanz.de/ilias.php?baseClass=ilrepositorygui&ref_id=2030138" 

print("Fetching the course page...")
# We use the 'session' we created in the last cell to get the page securely
response = session.get(COURSE_URL)

if response.status_code == 200:
    # Pass the HTML content to BeautifulSoup so we can search it
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Let's extract the title of the webpage just to verify we are in the right place!
    page_title = soup.title.text.strip() if soup.title else "No title found"
    print(f"✅ Successfully loaded the page!")
    print(f"Page Title: {page_title}")
    
else:
    print(f"❌ Failed to load the page. Status Code: {response.status_code}")

Fetching the course page...
✅ Successfully loaded the page!
Page Title: Inhalt: Applied time series analysis: ILIAS :: E-Learning Universität Konstanz


## Scraping the Links

In [6]:
import urllib.parse

print("Filtering for course materials...")

all_links = soup.find_all('a')
BASE_URL = "https://ilias.uni-konstanz.de/"

files_to_download = []
folders_to_explore = []

for link in all_links:
    text = link.get_text(strip=True)
    href = link.get('href')
    
    if not text or not href:
        continue
        
    # This makes sure relative links (like "ilias.php?...") become full links
    full_url = urllib.parse.urljoin(BASE_URL, href)
    
    # 1. Is it a file?
    if "cmd=sendfile" in full_url:
        files_to_download.append((text, full_url))
        
    # 2. Is it a folder?
    elif "/fold/" in full_url:
        folders_to_explore.append((text, full_url))

print(f"✅ Found {len(files_to_download)} files and {len(folders_to_explore)} folders!\n")

print("--- FILES ---")
for name, link in files_to_download:
    print(f"📄 {name}")

print("\n--- FOLDERS ---")
for name, link in folders_to_explore:
    print(f"📁 {name}")

Filtering for course materials...
✅ Found 4 files and 3 folders!

--- FILES ---
📄 Read me first: Applied Time Series Analysis in Summer 2026! (last revised: March 24, 2026)
📄 Lecture Plan Applied Time Series Analysis (last revised: March 31, 2026)
📄 pdf of Python Notes (by Kevin Sheppard), in case link does not work
📄 Time Series Books

--- FOLDERS ---
📁 Lectures
📁 Tutorials
📁 Programming Assignments


## Donwload Files

In [7]:
import os
import re

# 1. Define where you want to save the files on your computer
# This will create a folder right next to your Jupyter notebook
COURSE_FOLDER = "Applied_Time_Series_Analysis"
os.makedirs(COURSE_FOLDER, exist_ok=True)

print(f"Saving files to folder: {COURSE_FOLDER}\n")

for name, link in files_to_download:
    print(f"Fetching: {name}...")
    
    # Actually request the file download from ILIAS
    file_response = session.get(link)
    
    # 2. Extract the real filename from the server's response headers
    # ILIAS usually sends a header called 'Content-Disposition' that contains the exact file name
    content_disposition = file_response.headers.get('content-disposition', '')
    
    real_filename = None
    if 'filename=' in content_disposition:
        # A little regex magic to pull the filename out of the quotes
        found_names = re.findall('filename="?([^"]+)"?', content_disposition)
        if found_names:
            real_filename = found_names[0]
            
    # Fallback just in case the server doesn't provide a clean name
    if not real_filename:
        # Remove characters that Windows/Mac hate in filenames (like colons :)
        safe_name = re.sub(r'[\\/*?:"<>|]', "", name)
        real_filename = f"{safe_name}.pdf" 
        
    # Create the full path (e.g., Applied_Time_Series_Analysis/Lecture_Plan.pdf)
    file_path = os.path.join(COURSE_FOLDER, real_filename)
    
    # 3. Check if we already have it to avoid re-downloading!
    if not os.path.exists(file_path):
        with open(file_path, 'wb') as f:
            f.write(file_response.content)
        print(f"  ✅ Saved new file: {real_filename}\n")
    else:
        print(f"  ⏭️ Already exists. Skipping: {real_filename}\n")

print("Download run complete!")

Saving files to folder: Applied_Time_Series_Analysis

Fetching: Read me first: Applied Time Series Analysis in Summer 2026! (last revised: March 24, 2026)...
  ✅ Saved new file: Read me first_ Applied Time Series Analysis in Summer 2026! (last revised_ March 24, 2026)   .pdf

Fetching: Lecture Plan Applied Time Series Analysis (last revised: March 31, 2026)...
  ✅ Saved new file: Lecture Plan Applied Time Series Analysis (last revised_ March 31, 2026) .pdf

Fetching: pdf of Python Notes (by Kevin Sheppard), in case link does not work...
  ✅ Saved new file: pdf of Python Notes (by Kevin Sheppard), in case link does not work.pdf

Fetching: Time Series Books...
  ✅ Saved new file: Time Series Books.pdf

Download run complete!


## Donwload Folders

In [9]:
import os
import re
import urllib.parse
from bs4 import BeautifulSoup

def sync_ilias_folder(url, local_dir, visited_urls=None):
    # Initialize the memory on the very first run
    if visited_urls is None:
        visited_urls = set()
        
    # If we have already scanned this exact link, stop and turn back
    if url in visited_urls:
        return
    
    # Remember this URL so we don't scan it again
    visited_urls.add(url)
    
    print(f"\n📂 Scanning: {local_dir}")
    os.makedirs(local_dir, exist_ok=True)
    
    # Open the folder page
    response = session.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    BASE_URL = "https://ilias.uni-konstanz.de/"
    
    # Look at every link on this page
    for link in soup.find_all('a'):
        text = link.get_text(strip=True)
        href = link.get('href')
        
        if not text or not href or len(text) < 2:
            continue
            
        full_url = urllib.parse.urljoin(BASE_URL, href)
        
        # --- 1. Did we find a File? ---
        if "cmd=sendfile" in full_url:
            file_response = session.get(full_url)
            content_disposition = file_response.headers.get('content-disposition', '')
            
            real_filename = None
            if 'filename=' in content_disposition:
                found_names = re.findall('filename="?([^"]+)"?', content_disposition)
                if found_names:
                    real_filename = found_names[0]
                    
            if not real_filename:
                safe_name = re.sub(r'[\\/*?:"<>|]', "", text)
                real_filename = f"{safe_name}.pdf" 
                
            file_path = os.path.join(local_dir, real_filename)
            
            if not os.path.exists(file_path):
                print(f"  ⬇️ Downloading new file: {real_filename}")
                with open(file_path, 'wb') as f:
                    f.write(file_response.content)
            else:
                print(f"  ⏭️ Already exists: {real_filename}")
                
        # --- 2. Did we find a Sub-Folder? ---
        elif "/fold/" in full_url and full_url not in visited_urls:
            # Clean up the folder name for Windows/Mac
            safe_folder_name = re.sub(r'[\\/*?:"<>|]', "", text)
            new_local_dir = os.path.join(local_dir, safe_folder_name)
            
            # The Magic: The function calls ITSELF to dive deeper!
            sync_ilias_folder(full_url, new_local_dir, visited_urls)


# --- Run the Master Function ---
# We use the original COURSE_URL from cell 3
MAIN_COURSE_FOLDER = "Applied_Time_Series_Analysis"

print("🚀 Starting Deep Sync of the entire course...")
sync_ilias_folder(COURSE_URL, MAIN_COURSE_FOLDER)
print("\n🎉 Sync Complete! All folders and files are up to date.")

🚀 Starting Deep Sync of the entire course...

📂 Scanning: Applied_Time_Series_Analysis
  ⏭️ Already exists: Read me first_ Applied Time Series Analysis in Summer 2026! (last revised_ March 24, 2026)   .pdf
  ⏭️ Already exists: Lecture Plan Applied Time Series Analysis (last revised_ March 31, 2026) .pdf

📂 Scanning: Applied_Time_Series_Analysis\Lectures
  ⬇️ Downloading new file: Lecture Slides - Part 1 (last revised_ March 31, 2026).pdf

📂 Scanning: Applied_Time_Series_Analysis\Lectures\Python code snippets
  ⬇️ Downloading new file: GDPC1.csv
  ⬇️ Downloading new file: USRealGDP26.py
  ⬇️ Downloading new file: ATSAdata.zip
  ⬇️ Downloading new file: Lecture Slides - References (last revised_ March 31, 2026).pdf

📂 Scanning: Applied_Time_Series_Analysis\Tutorials

📂 Scanning: Applied_Time_Series_Analysis\Programming Assignments
  ⏭️ Already exists: pdf of Python Notes (by Kevin Sheppard), in case link does not work.pdf
  ⏭️ Already exists: Time Series Books.pdf

🎉 Sync Complete! All f